In [1]:
import pandas as pd

data = pd.read_csv('../Dataset/Scored_Cleaned_Data.csv')

In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

def evaluate_kmeans_custom_logic(df, target_col='Risk_score', n_folds=5):
    X_df = df.drop(columns=["Timestamp", "Group"], errors='ignore')
    spend_values = df[target_col].values
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    overall_results = []

    print(f"{'K':<5} | {'Avg Silhouette':<15} | {'Custom Deviation':<20}")
    print("-" * 50)

    for k in range(3, 20):
        fold_silhouettes = []
        fold_custom_deviations = []
        
        for train_idx, val_idx in kf.split(X_df):
            X_fold = X_df.iloc[train_idx]
            spend_fold = spend_values[train_idx]
            
            model = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = model.fit_predict(X_fold)
            
            # Silhouette
            sil = silhouette_score(X_fold, labels)
            fold_silhouettes.append(sil)
            
            # (sum(|M - means|^0.5))^2
            cluster_means = [spend_fold[labels == cid].mean() for cid in range(k) if np.any(labels == cid)]
            c_means = np.array(cluster_means)
            M = np.mean(c_means)
            diffs = np.abs(M - c_means)
            custom_dev = (np.sum(diffs**0.5))**2
            fold_custom_deviations.append(custom_dev)

        avg_sil = np.mean(fold_silhouettes)
        avg_dev = np.mean(fold_custom_deviations)
        overall_results.append({'k': k, 'Silhouette': avg_sil, 'Custom_Deviation': avg_dev})
        print(f"{k:<5} | {avg_sil:<15.4f} | {avg_dev:<20.2f}")

    return pd.DataFrame(overall_results)

kmeans_custom_results = evaluate_kmeans_custom_logic(data)

K     | Avg Silhouette  | Custom Deviation    
--------------------------------------------------
3     | 0.4799          | 11.45               
4     | 0.4184          | 40.83               
5     | 0.3753          | 102.32              
6     | 0.3732          | 430.03              
7     | 0.3801          | 1052.68             
8     | 0.3908          | 1622.51             
9     | 0.4099          | 2610.81             
10    | 0.4102          | 3324.30             
11    | 0.4137          | 4192.68             
12    | 0.4165          | 4897.31             
13    | 0.4141          | 5757.65             
14    | 0.4207          | 6672.71             
15    | 0.4255          | 7485.31             
16    | 0.4414          | 8580.32             
17    | 0.4438          | 9113.88             
18    | 0.4581          | 9872.94             
19    | 0.4790          | 10147.79            
